<a href="https://colab.research.google.com/github/abuhussein1504/NYC-Taxi-Trip-Duration/blob/main/NYC_Taxi_Trip_Duration_Modeling_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

TRAIN_PATH = r"/content/drive/MyDrive/NYC Taxi Trip Duration/split/train.csv"
VAL_PATH = r"/content/drive/MyDrive/NYC Taxi Trip Duration/split/val.csv"
TEST_PATH = r"/content/drive/MyDrive/NYC Taxi Trip Duration/split/test.csv"
SUBMISSION_OUTPUT_PATH = "submission.csv"

RANDOM_STATE = 42


# Feature engineering
def extract_datetime_features(df, datetime_cols):
    df = df.copy()
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        prefix = col.replace("_datetime", "") if "_datetime" in col else col

        df[f"{prefix}_dayofweek"] = df[col].dt.dayofweek
        df[f"{prefix}_month"] = df[col].dt.month
        df[f"{prefix}_hour"] = df[col].dt.hour
        df[f"{prefix}_dayofyear"] = df[col].dt.dayofyear
    return df


def haversine_miles(lat1, lat2, lon1, lon2):
    R = 3958.8
    lat1, lat2, lon1, lon2 = map(np.radians, [lat1, lat2, lon1, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return c * R


def engineer_features(df):
    df = extract_datetime_features(df, ["pickup_datetime"])
    df["distance"] = haversine_miles(
        df["pickup_latitude"], df["dropoff_latitude"],
        df["pickup_longitude"], df["dropoff_longitude"],
    )
    df["store_and_fwd_flag"] = (df["store_and_fwd_flag"] == "Y").astype(int)
    return df


def prepare_model_frame(df, drop_cols, fill_medians=None):
    df = df.drop(columns=drop_cols).drop_duplicates()
    medians = fill_medians if fill_medians is not None else df.median(numeric_only=True)
    df = df.fillna(medians)
    df["distance"] = np.log1p(df["distance"])
    return df, medians


# Load and engineer
train_raw = pd.read_csv(TRAIN_PATH)
val_raw = pd.read_csv(VAL_PATH)
test_raw = pd.read_csv(TEST_PATH)

train = engineer_features(train_raw)
val = engineer_features(val_raw)
test = engineer_features(test_raw)

train_copy, train_medians = prepare_model_frame(train, drop_cols=["id", "pickup_datetime"])
val_copy, _ = prepare_model_frame(val, drop_cols=["id", "pickup_datetime"], fill_medians=train_medians)
test_copy, _ = prepare_model_frame(test, drop_cols=["id", "pickup_datetime"], fill_medians=train_medians)

# Build feature matrix / target
X_train = train_copy.drop(columns=["trip_duration"])
y_train = np.log1p(train_copy["trip_duration"])

X_val = val_copy.drop(columns=["trip_duration"])
y_val = np.log1p(val_copy["trip_duration"])

X_test = test_copy.drop(columns=["trip_duration"])
y_test = np.log1p(test_copy["trip_duration"])

# One-hot encode categoricals + scale numeric columns - fit once on train, transform elsewhere
numeric_features = ["pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude"]
categorical_features = ["pickup_dayofweek", "pickup_month", "pickup_hour", "passenger_count"]

column_transformer = ColumnTransformer([
    ("ohe", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("scaling", StandardScaler(), numeric_features),
], remainder="passthrough")

X_train_transformed = column_transformer.fit_transform(X_train)
X_val_transformed = column_transformer.transform(X_val)
X_test_transformed = column_transformer.transform(X_test)
print("ColumnTransformer done.")

# Ridge, tuned via RidgeCV
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
ridge_cv.fit(X_train_transformed, y_train)
print("Best alpha:", ridge_cv.alpha_)

ridge_model = Ridge(alpha=ridge_cv.alpha_)
ridge_model.fit(X_train_transformed, y_train)

ridge_train_r2 = r2_score(y_train, ridge_model.predict(X_train_transformed))
ridge_val_r2 = r2_score(y_val, ridge_model.predict(X_val_transformed))
print(f"Ridge Train R²: {ridge_train_r2:.4f}")
print(f"Ridge Validation R²: {ridge_val_r2:.4f}")

cv_scores = cross_val_score(ridge_model, X_train_transformed, y_train, cv=5)
print("Ridge Cross-Validation Scores:", cv_scores)
print("Ridge Mean CV Score:", np.mean(cv_scores))

# XGBoost, for comparison
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
)
xgb_model.fit(X_train_transformed, y_train)

xgb_train_r2 = r2_score(y_train, xgb_model.predict(X_train_transformed))
xgb_val_r2 = r2_score(y_val, xgb_model.predict(X_val_transformed))
print(f"XGB Train R²: {xgb_train_r2:.4f}")
print(f"XGB Validation R²: {xgb_val_r2:.4f}")

# Final held-out check (only look at this once, after model selection)
best_model = xgb_model if xgb_val_r2 >= ridge_val_r2 else ridge_model
best_name = "XGB" if best_model is xgb_model else "Ridge"
test_r2 = r2_score(y_test, best_model.predict(X_test_transformed))
print(f"\nSelected model: {best_name}")
print(f"{best_name} Held-out Test R²: {test_r2:.4f}")

# Generate the actual Kaggle submission
submission_log_preds = best_model.predict(X_test_transformed)
submission_preds = np.expm1(submission_log_preds)  # back out of log space

submission = pd.DataFrame({
    "id": test_raw.loc[test_copy.index, "id"],
    "trip_duration": submission_preds,
})
submission.to_csv(SUBMISSION_OUTPUT_PATH, index=False)
print(f"\nSubmission written to {SUBMISSION_OUTPUT_PATH}")

ColumnTransformer done.
Best alpha: 1000.0
Ridge Train R²: 0.5934
Ridge Validation R²: 0.5931
Ridge Cross-Validation Scores: [0.59276167 0.59535043 0.49738152 0.59526727 0.59374062]
Ridge Mean CV Score: 0.5749003039467998
XGB Train R²: 0.7596
XGB Validation R²: 0.7363

Selected model: XGB
XGB Held-out Test R²: 0.7410

Submission written to submission.csv
